# RAPIDS:XGBOOST + SKFold10 + Regression Class Cutoff

- I trusted my local CV to select the two final models
- This is the best model that achieved with Optuna during the competition

## Useful imports

In [ ]:
## Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

## Import RAPIDS/CUDF
import cudf, cuml, cupy
from cudf.core.dataframe import DataFrame as cu_df
from cudf.core.series import Series as cu_series
print('RAPIDS',cudf.__version__)

In [ ]:
## Import important libs
import numpy as np
import pandas as pd
import os
from functools import partial
import scipy as sp

## Import CV and Metric
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

## Import Model
import xgboost as xgb

## Import random to seed and made reproducible
import random

## Hyperparameters

In [ ]:
SEED = 42
NUM_FOLDS = 10
TRAIN_PATH = '/kaggle/input/playground-series-s3e5/train.csv'
TEST_PATH = '/kaggle/input/playground-series-s3e5/test.csv'
SAMPLE_SUBMISSION = '/kaggle/input/playground-series-s3e5/sample_submission.csv'

## Seed Everything for Reproducibility

In [ ]:
def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(seed=SEED)

## Open the train/test/sample_submission

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION)

## Understand the data :)

In [ ]:
train.describe()

## Get what is useful for train the model

In [ ]:
X = train.drop(columns=['quality', 'Id'])
features = X.columns
X_test = test.drop(columns=['Id'])
y = train.quality

In [ ]:
X.describe()

In [ ]:
X_test.describe()

## Optimized Rounder 

In [ ]:
## I posted some references on the discussion of where I got this snippet of code, but right now I dont remember exactly the notebook (btw thanks for the implementation!)

class OptimizedRounder(object):
    def __init__(self):
        self.coef_ = 0

    def _kappa_loss(self, coef, X, y):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 3
            elif pred >= coef[0] and pred < coef[1]:
                X_p[i] = 4
            elif pred >= coef[1] and pred < coef[2]:
                X_p[i] = 5
            elif pred >= coef[2] and pred < coef[3]:
                X_p[i] = 6
            elif pred >= coef[3] and pred < coef[4]:
                X_p[i] = 7
            else:
                X_p[i] = 8

        ll = cohen_kappa_score(y, X_p, weights='quadratic')
        return -ll

    def fit(self, X, y):
        loss_partial = partial(self._kappa_loss, X=X, y=y)
        initial_coef = [3.5, 4.5, 5.5, 6.5, 7.5]
        self.coef_ = sp.optimize.minimize(loss_partial, initial_coef, method='nelder-mead')

    def predict(self, X, coef):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 3
            elif pred >= coef[0] and pred < coef[1]:
                X_p[i] = 4
            elif pred >= coef[1] and pred < coef[2]:
                X_p[i] = 5
            elif pred >= coef[2] and pred < coef[3]:
                X_p[i] = 6
            elif pred >= coef[3] and pred < coef[4]:
                X_p[i] = 7
            else:
                X_p[i] = 8
        return X_p.astype('int')

    def coefficients(self):
        return self.coef_['x']

## CV Function to help us

In [ ]:
def cross_valid(model, train, target, test, num_folds=10, random_state=42):

    train_oof = np.zeros((len(train)))
    test_preds = 0

    kf = StratifiedKFold(n_splits=num_folds, random_state=SEED, shuffle=True)
    scores = []
    
    params = {
        'random_state': SEED,
        'objective': 'reg:squarederror',
        'nthread': -1, 
        'tree_method': 'gpu_hist', 
        
        'lambda': 5.619263832415134e-06, 
        'alpha': 0.004114579891733313, 
        'max_depth': 5, 
        'eta': 0.04855597008155139, 
        'gamma': 4.4150816096917164e-07, 
        'min_child_weight': 20, 
        'subsample': 0.4923174680251535, 
        'colsample_bytree': 0.7120320651725672, 
        'max_delta_step': 44.336096164825314, 
        'num_boost_round': 1000
     }
    
    num_rounds = params['num_boost_round']
    
    xgb_train_preds = np.zeros(len(train.index), )
    
    coef = []
    

    for f, (train_ind, val_ind) in enumerate(kf.split(train, target)):

        train_df, val_df = train.iloc[train_ind][columns], train.iloc[val_ind][columns]
        
        train_target, val_target = target[train_ind], target[val_ind]
        

        xgb_x_train = pd.DataFrame(train_df)
        xgb_x_valid = pd.DataFrame(val_df)

        xgb_x_train_cudf = cu_df(xgb_x_train)
        y_train_cudf = cu_series(train_target)
        xgb_x_valid_cudf = cu_df(xgb_x_valid)
        y_valid_cudf = cu_series(val_target)

        trn_data = xgb.DMatrix(xgb_x_train_cudf, label=y_train_cudf)
        val_data = xgb.DMatrix(xgb_x_valid_cudf, label=y_valid_cudf)

        model = xgb.train(params, 
        trn_data,
        num_rounds,
        evals = [(val_data, "val_data")], 
        verbose_eval=False, 
        early_stopping_rounds=50
        )

        xgb_valid_preds = model.predict(xgb.DMatrix(xgb_x_valid_cudf), ntree_limit=model.best_iteration)
        
        optR = OptimizedRounder()
        optR.fit(xgb_valid_preds, val_target)
                
        temp_oof = optR.predict(xgb_valid_preds, optR.coefficients())
        
        train_oof[val_ind] = temp_oof
    
        test_oof_preds = model.predict(xgb.DMatrix(test[columns]), ntree_limit=model.best_iteration)
        
        test_oof_preds = optR.predict(test_oof_preds, optR.coefficients())
        
        coef.append(optR.coefficients())
        
        print(optR.coefficients())
        
        test_preds += test_oof_preds/num_folds
        
        scores.append(cohen_kappa_score(val_target, temp_oof, weights='quadratic'))
        
        print("Fold " , f, " ", cohen_kappa_score(val_target, temp_oof, weights="quadratic"))
        
                                
    print("Mean Kappa Score: ", np.mean(scores))
    print("Kappa Score OOF: ", cohen_kappa_score(y, train_oof, weights='quadratic'))

    return train_oof, test_preds, np.mean(scores), coef

## Call the CV function

In [ ]:
## Here I can select which features I want to use
columns = features

## If you want to do ensemble is good to save the oof train / preds
train_oof_1, test_preds_1, score_oof_1, coef = cross_valid(None, X, y, X_test, num_folds=NUM_FOLDS, random_state=SEED)

## Save oof and preds

In [ ]:
np.save('train_oof_xgb.npy', train_oof_1)
np.save('test_preds_xgb.npy', test_preds_1)

## Update Sample Submission and create CSV file

In [ ]:
sample_submission['quality'] = test_preds_1.round().astype(int)
sample_submission.to_csv('submission.csv', index=False)
sample_submission